<a href="https://colab.research.google.com/github/idialloaka-ai/DAILYCHALLENGE/blob/master/Daily_challenge_W6D5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Daily Challenge: Reliable BERT Analysis
This notebook covers data loading, fine-tuning DistilBERT for sentiment analysis, performance evaluation, and attention visualization.

In [ ]:
!pip install -q datasets transformers[torch] evaluate accelerate

### 1. Data Loading and Inspection

In [ ]:
from datasets import load_dataset
import pandas as pd

# Load the sentiment configuration of tweet_eval
dataset = load_dataset("tweet_eval", "sentiment")

# Display class distribution
print("Class Distribution:")
for split in dataset.keys():
    df_split = pd.DataFrame(dataset[split])
    print(f"{split}: {df_split['label'].value_counts().to_dict()}")

# Save two examples per label for later visualization
examples = {}
for label in [0, 1, 2]: # 0: Negative, 1: Neutral, 2: Positive
    examples[label] = dataset['test'].filter(lambda x: x['label'] == label).select(range(2))

print("\nExamples saved for task 5 inspection.")

### 2. Tokenization Pipeline

In [ ]:
from transformers import AutoTokenizer

model_ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

def preprocess_function(examples):
    # Tokenize tweets with truncation and padding
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

# Map the dataset
tokenized_dataset = dataset.map(preprocess_function, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["text"])
tokenized_dataset.set_format("torch")

### 3. Fine-tuning

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

# Load metrics
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "f1": f1}

# Load model with 3 labels
model = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=3)

training_args = TrainingArguments(
    output_dir="./distilbert-sentiment",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics,
)

# Train the model
trainer.train()

### 4. Evaluation and Calibration

In [ ]:
import torch
import matplotlib.pyplot as plt
import torch.nn.functional as F

# Evaluate on test set
results = trainer.predict(tokenized_dataset["test"])
print(f"Test Results: {results.metrics}")

# Get softmax scores
logits = torch.from_numpy(results.predictions)
probs = F.softmax(logits, dim=-1)
confidences, _ = torch.max(probs, dim=-1)

# Plot confidence histogram
plt.figure(figsize=(10, 6))
plt.hist(confidences.numpy(), bins=10, range=(0, 1), color='skyblue', edgecolor='black')
plt.title("Confidence Score Distribution (Calibration Analysis)")
plt.xlabel("Confidence Score")
plt.ylabel("Number of Samples")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

print("Observation: High peaks near 1.0 indicate high model confidence, while spread across middle bins suggests potential under-confidence or uncertainty in complex tweets.")

### 5. Attention Inspection

In [ ]:
import seaborn as sns

# Select one saved negative example
test_text = examples[0][0]['text']
inputs = tokenizer(test_text, return_tensors="pt").to(model.device)

# Get attention weights (output_attentions=True)
outputs = model(**inputs, output_attentions=True)
# Last layer attention: shape [batch, heads, seq_len, seq_len]
last_layer_attn = outputs.attentions[-1][0].detach().cpu()

# Mean across all heads
avg_attn = last_layer_attn.mean(dim=0)

# Extract attention from [CLS] (index 0) to all tokens
cls_attn = avg_attn[0, :].numpy()
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

# Visualize
plt.figure(figsize=(12, 4))
sns.barplot(x=tokens, y=cls_attn, palette="viridis")
plt.title(f"Attention weights for [CLS] token (Tweet: {test_text})")
plt.xticks(rotation=45)
plt.show()

print("Observation: The model assigns higher attention weights to sentiment-bearing tokens to compute the final [CLS] representation used for classification.")